# Aprendizaje por diferencia temporal (TD)

En este ejercicio vamos a implementar el método de diferencia temporal para para resolver MDPs con transiciones y recompensas desconocidas. 

El método de TD se basa en el cálculo de los valores para los estados de acuerdo con la fórmula:

$V^\pi(s) \leftarrow (1-\alpha)V^\pi(s) + \alpha[R(s, \pi(s), s') + \gamma V^\pi(s')]$

donde $\alpha$ corresponde a la taza de aprendizaje.

#### Task 1

Para implementar TD definimos `td_learning.py` como una extensión del ambiente de Gridworld. Dentro de esta extensión debemos asegurarnos que:
  - Seguimos una política, dada como un parámetro del ambiente.
  - Cada paso de la muestra ejecuta la política para el estado actual, obteniendo un estado de llegada y una recompensa. Tenga en cuenta que las acciones no son determinísticas y la ejecución de cada acción depende de un factor de ruido (en el caso de Gridworld, tomaremos un factor de ruido de 0.2 para las acciones abajo e izquierda y 0.3 para las acciones arriba y derecha, desconocida para el agente). Por ejemplo, el agente tiene una probabilidad de 0.8 de moverse a la izquierda y abajo y terminar en el estado correspondiente y probabilidad de 0.2 de terminar en cualquiera de las otras tres direcciones.
  - A partir de los valores obtenidos de diferentes muestras, obtenga una nueva política.
  - Utilice una taza de aprendizaje de `0.7`

Responda las preguntas
1. ¿Cuántas iteraciones son necesarias para que la política de las muestras se estabilice?
2. ¿Cómo se compara la política obtenida con la calculada utilizando iteración de valores o iteración de políticas? ¿Existe alguna diferencia? ¿Porqué?



In [ ]:
 # Enviroment and MDP definition

import random

class EnvironmentNuevo:
    def __init__(self, board, exit_state):
        self.board = board
        self.nrows = len(board)
        self.ncols = len(board[0]) 
        self.initial_state = self._find_initial_state()
        self.current_state = self.initial_state
        self.actions = ['up', 'right', 'down', 'left', 'exit']
        # self.P = self._build_transition_matrix()
        self.exit_state = exit_state
        
    def _build_transition_matrix(self):
        nA = len(self.actions)
    
        P = [[[[0 for _ in range(nA)] for _ in range(nA)]
              for _ in range(self.ncols)]
              for _ in range(self.nrows)]
    
        for i in range(self.nrows):
            for j in range(self.ncols):
            
                if self.board[i][j] == '#':
                    continue
                
                # Si es estado terminal
                if self._is_exit(i, j):
                    P[i][j][4][4] = 1.0  # exit → exit
                    continue
                
                for a in range(4):  # solo up,right,down,left
                
                    clockwise = (a + 1) % 4
                    counterclock = (a - 1) % 4
    
                    P[i][j][a][a] = 0.6
                    P[i][j][a][clockwise] = 0.2
                    P[i][j][a][counterclock] = 0.1
                    P[i][j][a][a] += 0.1  # 10% quedarse = misma acción
    
        return P

    def _find_initial_state(self):
        for i in range(self.nrows):
            for j in range(self.ncols):
                if self.board[i][j] == 'S':
                    return (i, j)
        return (1, 1)


    def _is_exit(self, r, c):
        return self.exit_state == (r, c) or self.board[r][c] == '-100'

    def _move(self, r, c, action):
        dr, dc = 0, 0
        if action == 'up':
            dr = -1
        elif action == 'down':
            dr = 1
        elif action == 'left':
            dc = -1
        elif action == 'right':
            dc = 1

        nr, nc = r + dr, c + dc
        if nr < 0 or nr >= self.nrows or nc < 0 or nc >= self.ncols:
            return r, c
        if self.board[nr][nc] == '#':
            return r, c
        return nr, nc
   
    def _get_reward(self, r, c):
        if self._is_exit(r, c):
            return float(self.board[r][c])
        if self.board[r][c] == '#':
            return 0
        if self.board[r][c] == 'S':
            return -1
        return float(self.board[r][c])
    
    def get_current_state(self):
        return self.current_state

    def get_posible_actions(self, state):
        r, c = state
        if self._is_exit(r, c):
            return ['exit']
        if self.board[r][c] == '#':
            return []
        return ['up', 'right', 'down', 'left']
   
    def do_action(self, idx_action):
        r, c = self.current_state
    
        # Si es terminal
        if self._is_exit(r, c):
            if idx_action == 4:
                return float(self.board[r][c]), self.current_state
            return 0, self.current_state
    
        # Elegir acción real 
        real_action = self.actions[idx_action]
    
        nr, nc = self._move(r, c, real_action)
        self.current_state = (nr, nc)
        
    
        reward = self._get_reward(nr, nc)
    
        return reward, self.current_state

    def reset(self):
        self.current_state = self.initial_state

    def is_terminal(self):
        r, c = self.current_state
        return self._is_exit(r, c)
    

env=EnvironmentNuevo([
    ['-1']*12,
    ['-1']*12,
    ['-1']*12,
    ['-1']*12,
    ['S'] + ['-100']*10 + ['100']
], exit_state=(4, 11))

In [ ]:
import numpy as np

class SARSA():
    def __init__(self, env, Q, alpha=0.81, gamma=0.96, epsilon=0.9):
        self.env = env
        self.Q = Q
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    
    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            #exploration
            return np.random.randint(0, len(self.env.get_posible_actions(state)))
        else:
            #explotation
            return np.argmax(self.Q[state])
        
    def action_function(self, state1, action1, reward, state2, action2):
        #$Q(s,a) \leftarrow (1-\alpha)Q(s,a) + \alpha[R(s) + \gamma Q(s',a')] $
        self.Q[state1][action1] = (1 - self.alpha) * self.Q[state1][action1] + self.alpha * (reward + self.gamma * self.Q[state2][action2])
        
    
 